# Native PyTorch tutorial 8

Two 50,000-epoch fits: spatial Sobolev supervision (coefficient 1e-3) versus value-only training. Temporal derivatives are measured, not supervised. Predictions, derivatives and superresolution comparisons are all converted back to physical units.

Run all cells in order. Full scientific workloads are the default; `NIF_SMOKE=1` is a separate development run and does not establish full convergence. The original TensorFlow and `_colab` notebooks are preserved as legacy references.

In [ ]:
# Run this cell first in a clean Colab Python 3.11–3.13 session.
import os, sys, subprocess, importlib.util
from pathlib import Path
# A local checkout is used directly for development; Colab installs the public pytorch branch.
repo = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "nif" / "model.py").exists()), None)
if repo is not None:
    sys.path.insert(0, str(repo))
else:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet",
        "torch==2.14.1", "numpy==2.1.3", "scipy==1.16.3", "matplotlib==3.10.0",
        "protobuf==7.36.2", "google-crc32c==1.9.0", "tfrecord==1.14.6",
        "git+https://github.com/pswpswpsw/nif.git@pytorch"])
import copy, json, time, math
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from matplotlib import pyplot as plt
import nif
# FULL is the default. Set NIF_SMOKE=1 before starting only for a short development check.
SMOKE = os.environ.get("NIF_SMOKE", "0") == "1"
device = torch.device(os.environ.get("NIF_DEVICE", "cuda" if torch.cuda.is_available() else "cpu"))
torch.manual_seed(42)
np.random.seed(42)
if device.type == "cpu":
    torch.set_num_threads(min(4, os.cpu_count() or 1))
print({"torch": torch.__version__, "device": str(device), "mode": "SMOKE" if SMOKE else "FULL"})

OUT = Path(os.environ.get("NIF_OUTPUT_DIR", "pytorch_outputs")) / "tutorial8"
OUT.mkdir(parents=True, exist_ok=True)


In [ ]:
# Explicit native loops replace compile/fit/callbacks. The loss reduction is per row,
# then the mean over the batch; area weights multiply each row's MSE (not weight-normalized).
def dense_tanh(inputs, outputs):
    # Preserve the original Dense default: Glorot uniform kernel and zero bias.
    layer = nn.Linear(inputs, outputs)
    nn.init.xavier_uniform_(layer.weight)
    nn.init.zeros_(layer.bias)
    return nn.Sequential(layer, nn.Tanh())

def epochs(full):
    return 2 if SMOKE else full

def tensor(array, on_device=device):
    return torch.as_tensor(array, dtype=torch.float32, device=on_device)

def loader(features, target, batch_size, weight=None):
    arrays = [tensor(features, "cpu"), tensor(target, "cpu")]
    if weight is not None:
        arrays.append(tensor(weight, "cpu").reshape(-1))
    return DataLoader(TensorDataset(*arrays), batch_size=batch_size, shuffle=True,
                      drop_last=False, pin_memory=device.type == "cuda")

def predict(model, features, batch_size=1024):
    model.eval()
    result = []
    with torch.no_grad():
        for xb in torch.as_tensor(features, dtype=torch.float32).split(batch_size):
            result.append(model(xb.to(next(model.parameters()).device)).detach().cpu())
    return torch.cat(result).numpy()

def centralize_gradients(model):
    # TF kernels are [input,output]; native Linear stores [output,input].
    # Centralize along the input axes, preserving the original mathematical operation.
    for module in model.modules():
        for name, parameter in module.named_parameters(recurse=False):
            if parameter.grad is not None and parameter.ndim > 1:
                axes = tuple(range(1, parameter.ndim)) if isinstance(module, nn.Linear) else tuple(range(parameter.ndim-1))
                parameter.grad.sub_(parameter.grad.mean(dim=axes, keepdim=True))

def save_checkpoint(model, optimizer, path, epoch):
    payload = {"model_state": {k: v.detach().cpu() for k,v in model.state_dict().items()},
               "optimizer_state": optimizer.state_dict(), "epoch": epoch,
               "framework": "pytorch", "smoke": SMOKE}
    if hasattr(model, "cfg_shape_net"):
        payload.update(model_class=type(model).__name__, cfg_shape_net=model.cfg_shape_net,
                       cfg_parameter_net=model.cfg_parameter_net, mixed_policy="float32")
    torch.save(payload, path)

def save_and_check(model, optimizer, name, example, factory=None):
    path = OUT / (name + ".pt")
    save_checkpoint(model, optimizer, path, -1)
    checkpoint = torch.load(path, map_location=device, weights_only=True)
    if factory is None:
        factory = lambda: type(model)(copy.deepcopy(model.cfg_shape_net),
                                       copy.deepcopy(model.cfg_parameter_net), "float32")
    restored = factory().to(device)
    restored.load_state_dict(checkpoint["model_state"])
    restored.eval(); model.eval()
    with torch.no_grad():
        expected, actual = model(tensor(example)), restored(tensor(example))
    torch.testing.assert_close(actual, expected, rtol=1e-5, atol=1e-6)
    print("Checkpoint predictions verified:", path)
    return restored

def train(model, batches, optimizer, nepoch, name, schedule=None,
          centralization=False, loss_function=None, before_epoch=None):
    history = []
    start = time.monotonic()
    for epoch in range(nepoch):
        model.train()
        if schedule is not None:
            for group in optimizer.param_groups:
                group["lr"] = schedule(epoch, group["lr"])
        if before_epoch is not None:
            before_epoch(epoch)
        summed, seen = {}, 0
        for batch in batches:
            xb, yb = (item.to(device) for item in batch[:2])
            optimizer.zero_grad(set_to_none=True)
            if loss_function is None:
                per_row = (model(xb) - yb).square().mean(dim=-1)
                if len(batch) == 3:
                    per_row = per_row * batch[2].to(device).reshape(-1)
                data_loss = per_row.mean()
                values = {"mse": data_loss}
            else:
                data_loss, values = loss_function(model, xb, yb)
            regularization = model.regularization_loss(xb) if hasattr(model, "regularization_loss") else xb.new_zeros(())
            loss = data_loss + regularization
            if not torch.isfinite(loss):
                raise FloatingPointError(f"Non-finite loss in {name}, epoch {epoch}")
            loss.backward()
            if centralization:
                centralize_gradients(model)
            optimizer.step()
            for key, value in {**values, "regularization": regularization, "loss": loss}.items():
                summed[key] = summed.get(key, 0.0) + float(value.detach()) * len(xb)
            seen += len(xb)
        row = {"epoch": epoch, "rows": seen, **{key: value / seen for key,value in summed.items()}}
        history.append(row)
        if epoch % 100 == 0 or epoch == nepoch - 1:
            print(name, row, "elapsed_seconds", round(time.monotonic()-start, 2))
            (OUT / (name + "_history.json")).write_text(json.dumps(history, indent=2))
        if epoch % 1000 == 0 or epoch == nepoch - 1:
            save_checkpoint(model, optimizer, OUT / (name + "_latest.pt"), epoch)
    fig, ax = plt.subplots(figsize=(7, 3))
    ax.semilogy([row["loss"] for row in history]); ax.set(xlabel="epoch", ylabel="loss", title=name)
    fig.tight_layout(); fig.savefig(OUT / (name + "_loss.png")); plt.show(); plt.close(fig)
    return history

def wave_figure(truth, prediction, name):
    fig, axs = plt.subplots(1,3,figsize=(15,3))
    for ax, values, title in zip(axs, (truth,prediction,prediction-truth), ("true","pred","error")):
        im=ax.contourf(tt,xx,values.reshape(10,200),levels=50,cmap="seismic")
        fig.colorbar(im,ax=ax); ax.set(xlabel="t",ylabel="x",title=title)
    fig.tight_layout(); fig.savefig(OUT / (name+"_reconstruction.png")); plt.show(); plt.close(fig)

def write_metrics(values):
    print(json.dumps(values, indent=2))
    (OUT / "metrics.json").write_text(json.dumps({"mode": "smoke" if SMOKE else "full", **values}, indent=2))


# Recall

- Previously we have a challenge in [Tutorial 6](https://colab.research.google.com/github/pswpswpsw/nif/blob/master/tutorial/6_revisit_multi_scale_NIF_with_L1_L2_regularization.ipynb): we get spurious signals when we sample the signal on a finer grid. We tried standard regularization but it does not work.

- How about Sobolov training? - not just fit the value but also the derivative

# Take home message

- Sobolov training works for reducing the effect of spurious waves happened in super-resolution

# What you will learn

- how to perform Sobolov training with just a few lines of code


## First, let's recall our example

In [ ]:
NT=10 # 20
NX=200

x = np.linspace(0,1,NX,endpoint=False)
t = np.linspace(0,100,NT,endpoint=False)

xx,tt=np.meshgrid(x,t)

omega = 400
c = 0.12/20
x0 = 0.2

u = np.exp(-1000*(xx-x0-c*tt)**2)*np.sin(omega*(xx-x0-c*tt))

# vis
plt.figure()
for i in range(NT):
    plt.plot(x,u[i,:],'-',label=str(i) + '-th time')

plt.xlabel('$x$',fontsize=25)
plt.ylabel('$u$',fontsize=25)


# vis iso
plt.figure(figsize=(4,4))
ax = plt.axes(projection='3d')
ax.plot_surface(xx,tt,u,cmap="rainbow", lw=2)#,rstride=1, cstride=1)
ax.view_init(57, -80)
ax.set_xlabel(r'$x$',fontsize=25)
ax.set_ylabel(r'$t$',fontsize=25)
ax.set_zlabel(r'$u$',fontsize=25)

## Next, we get the ground true derivatives

In [ ]:
## get derivative data

dudx = np.exp(-1000*(xx-x0-c*tt)**2)*(-2000*(xx-x0-c*tt)*np.sin(omega*(xx-x0-c*tt)) + 
                                      omega*np.cos(omega*(xx-x0-c*tt)))

dudt = np.exp(-1000*(xx-x0-c*tt)**2)*(2000*c*(xx-x0-c*tt)* np.sin(omega*(xx-x0-c*tt)) - 
                                      omega*c* np.cos(omega*(xx-x0-c*tt)))


dudx_1d = dudx.reshape(-1,1)
dudt_1d = dudt.reshape(-1,1)

# Setup for NIF with SIREN

In [ ]:
cfg_shape_net = {"use_resblock": False, "connectivity": "full", "input_dim": 1,
                 "output_dim": 1, "units": 30, "nlayers": 2,
                 "weight_init_factor": 0.01, "omega_0": 30.0}
cfg_parameter_net = {"use_resblock": False, "input_dim": 1, "latent_dim": 1,
                     "units": 30, "nlayers": 2, "activation": "swish"}
nepoch,lr,batch_size=50000,1e-4,512


### Here, we will use the following loss function

$$
\mathcal{L} = \mathcal{L}_{mse} + \alpha \mathcal{L}_{mse, derivative} = \frac{1}{M} \sum_i (u(x_i,y_i) - u_{NN}(x_i,y_i;\theta))^2 + \alpha  \frac{1}{M} \sum_i (\frac{\partial u}{\partial x}(x_i,y_i) - \frac{\partial u_{NN}}{\partial x}(x_i,y_i;\theta))^2
$$

then we use Adam optimizer to find $\min_{\theta} \mathcal{L}$

We empirically choose $\alpha=10^{-3}$. Intuitively, this number is related to when will the optimizer turns its attention to minimize $\mathcal{L}_{mse, derivative}$

In [ ]:
coef_grad=1e-3


### get training data set

In [ ]:
from nif.demo import TravelingWaveHighFreq
tw = TravelingWaveHighFreq()
train_data = tw.data
train_dataset = loader(train_data[:, :2], train_data[:, 2:3], batch_size=512)
train_data_ng=tw.data
train_mean_ng,train_std_ng=tw.mean,tw.std


### Factor correction for the derivative

- Note that we normalised both coordinate $x$ and output $u$, so the derivative needs a factor correction

- Let's denote $\tilde{x}$ and $\tilde{u}$ as normalized version of $x$ and $u$. 
- Then we have
$$
\partial{\tilde{u}}/\partial{\tilde{x}} = \frac{\partial{{u}}/\partial{{x}}}{\sigma_u/\sigma_x}
$$
where $\sigma_u$ and $\sigma_x$ are the denominators for $\tilde{u}$ and $\tilde{x}$.

### Now, we augment the original dataset with derivative information

In [ ]:
train_data = np.hstack([tw.data, dudt_1d/(tw.std[2]/tw.std[0]), dudx_1d/(tw.std[2]/tw.std[1])])

train_mean = np.hstack([tw.mean,0,0])
train_std = np.hstack([tw.std, tw.std[2]/tw.std[0], tw.std[2]/tw.std[1]])

In [ ]:
train_dataset=loader(train_data[:,:2],train_data[:,2:5],batch_size)


### Use of `JacobianLayer`

- We use just a few lines of codes in the below to implement Sobolov training
- We only regularize the spatial smoothness of the solution using du/dx. Since we are more interested in super-resolution in space in this case.
- We also observe du/dt just to show that the **du/dt from trained NN is not trustworthy at all** if you don't supervise it to learn.

In [ ]:
from nif.layers import JacobianLayer

def sobolev_loss(model,xb,yb,supervise_spatial=True):
    # Full y plus derivatives ordered [t,x]; supervise ONLY spatial derivative.
    y,jac=JacobianLayer(model,[0],[0,1])(xb)
    field_mse=(y-yb[:,:1]).square().mean()
    temporal_mse=(jac[:,0,0]-yb[:,1]).square().mean()
    spatial_mse=(jac[:,0,1]-yb[:,2]).square().mean()
    loss=field_mse+coef_grad*spatial_mse if supervise_spatial else field_mse
    return loss,{"field_mse":field_mse,"temporal_derivative_mse":temporal_mse,
                 "spatial_derivative_mse":spatial_mse,"weighted_temporal_mse":coef_grad*temporal_mse,
                 "weighted_spatial_mse":coef_grad*spatial_mse}

model_opt=nif.NIFMultiScale(copy.deepcopy(cfg_shape_net),copy.deepcopy(cfg_parameter_net),"float32").to(device)
optimizer=torch.optim.Adam(model_opt.parameters(),lr=lr,eps=1e-7)
history_sobolev=train(model_opt,train_dataset,optimizer,epochs(nepoch),"with_grad",loss_function=sobolev_loss)
model_opt=save_and_check(model_opt,optimizer,"with_grad",train_data[:64,:2])


# Compare it with Vanilla NIF with SIREN

In [ ]:
model_opt_ng=nif.NIFMultiScale(copy.deepcopy(cfg_shape_net),copy.deepcopy(cfg_parameter_net),"float32").to(device)
optimizer_ng=torch.optim.Adam(model_opt_ng.parameters(),lr=lr,eps=1e-7)
history_baseline=train(model_opt_ng,train_dataset,optimizer_ng,epochs(nepoch),"without_grad",
                       loss_function=lambda m,x,y:sobolev_loss(m,x,y,False))
model_opt_ng=save_and_check(model_opt_ng,optimizer_ng,"without_grad",train_data[:64,:2])


## Check the reconstruction. 

### No visual difference for those values at 0-level

- But that does not mean the derivative will be nicely reconstructed

In [ ]:
def predict_derivatives(model,features):
    values=[]
    for xb in tensor(features).split(512):
        y,jac=JacobianLayer(model,[0],[0,1])(xb)
        values.append(torch.cat([y,jac[:,0,:]],dim=1).detach().cpu())
    return torch.cat(values).numpy()
pred_ng=predict_derivatives(model_opt_ng,train_data[:,:2])
pred_wg=predict_derivatives(model_opt,train_data[:,:2])
u_ng,u_wg=pred_ng[:,0].reshape(10,200),pred_wg[:,0].reshape(10,200)


In [ ]:
# Value and derivative diagnostics are displayed in consistent physical units.
physical_scale=np.array([tw.std[2],tw.std[2]/tw.std[0],tw.std[2]/tw.std[1]])
physical_mean=np.array([tw.mean[2],0,0])
truth_physical=train_data[:,2:5]*physical_scale+physical_mean
pred_ng_physical=pred_ng*physical_scale+physical_mean
pred_wg_physical=pred_wg*physical_scale+physical_mean
fig,axs=plt.subplots(3,2,figsize=(14,9))
for column,snapshot in enumerate((0,5)):
    take=slice(snapshot*200,(snapshot+1)*200)
    for component,label in enumerate(("u","du/dt","du/dx")):
        ax=axs[component,column]
        ax.plot(x,truth_physical[take,component],"k-",label="true")
        ax.plot(x,pred_ng_physical[take,component],"r--",label="without gradient")
        ax.plot(x,pred_wg_physical[take,component],"b-.",label="with gradient")
        ax.set(xlabel="physical x",ylabel=label,title=f"t={t[snapshot]}");ax.legend()
fig.tight_layout();fig.savefig(OUT / "physical_value_derivatives.png");plt.show();plt.close(fig)


### Major differences in 1-level: the derivatives

In [ ]:
dudt_ng,dudx_ng=pred_ng[:,1].reshape(10,200),pred_ng[:,2].reshape(10,200)
dudt_wg,dudx_wg=pred_wg[:,1].reshape(10,200),pred_wg[:,2].reshape(10,200)
print("physical derivative MSE [du/dt,du/dx], baseline:",np.mean((pred_ng_physical[:,1:]-truth_physical[:,1:])**2,axis=0))
print("physical derivative MSE [du/dt,du/dx], Sobolev:",np.mean((pred_wg_physical[:,1:]-truth_physical[:,1:])**2,axis=0))


### Neither fits du/dt well

### But the first one fits du/dx well, and you can see if not regularized, the prediction on du/dx can be wild even though 0-level is well-fitted.

# Now we turn attention to super-resolution

- let's sample 800 points in space (note in the training data only 200 points in space)

In [ ]:
x_new_res = 800

test_x = np.linspace(0,1,x_new_res,endpoint=False)
test_t = np.linspace(0,100,10,endpoint=False)

test_xx, test_tt = np.meshgrid(test_x, test_t)
u_test = np.exp(-1000*(test_xx-x0-c*test_tt)**2)*np.sin(omega*(test_xx-x0-c*test_tt))


test_xx_n = (test_xx-tw.mean[1])/tw.std[1]
test_tt_n = (test_tt-tw.mean[0])/tw.std[0]


test_data = np.vstack([test_tt_n.flatten(), test_xx_n.flatten()]).T

In [ ]:
# Correct inverse scaling: analytic u_test is physical, so predictions must be physical too.
u_ng_test=predict(model_opt_ng,test_data)[:,0].reshape(10,x_new_res)*tw.std[2]+tw.mean[2]
u_wg_test=predict(model_opt,test_data)[:,0].reshape(10,x_new_res)*tw.std[2]+tw.mean[2]
assert u_ng_test.shape==u_test.shape==u_wg_test.shape


### Super-resolution will show a lot of spurious waves - which can be alleivated by Solobov training

In [ ]:
for i in range(10):
    fig,axs=plt.subplots(1,2,figsize=(16,3))
    for ax,prediction,label in ((axs[0],u_ng_test,"without gradient"),(axs[1],u_wg_test,"with gradient")):
        ax.plot(test_x,prediction[i],label=label);ax.plot(test_x,u_test[i],"k-",label="true")
        ax.set(xlabel="physical x",ylabel="physical u",ylim=(-1.2,1.2),title=f"t={test_t[i]}");ax.legend()
    fig.tight_layout();fig.savefig(OUT / f"superresolution_{i}.png");plt.show();plt.close(fig)
write_metrics({"epochs_per_fit":epochs(nepoch),"fits":2,"spatial_loss_coefficient":coef_grad,
    "baseline_physical_component_mse":np.mean((pred_ng_physical-truth_physical)**2,axis=0).tolist(),
    "sobolev_physical_component_mse":np.mean((pred_wg_physical-truth_physical)**2,axis=0).tolist(),
    "baseline_physical_superresolution_mse":float(np.mean((u_ng_test-u_test)**2)),
    "sobolev_physical_superresolution_mse":float(np.mean((u_wg_test-u_test)**2))})


# Conclusion 

- Original SIREN work leads to undesireable super-resolution spurious wave issues

- Regularizing with gradient information can help spurious high wave signal in super resolution

- It is very easy to implement in `nif`